# ICT4403 mT5 English French
Kaggle run: https://www.kaggle.com/code/eltonlewis/ict4403-mt5-english-french (version 3).
This notebook was assembled from the Kaggle MCP source and execution log. The code cell is the script executed by Kaggle; stream outputs are the saved run log. Checkpoints and data files are separate Kaggle output artifacts.
Saved output files: data/processed/manifest.json, data/processed/test.jsonl, data/processed/train.jsonl, data/processed/validation.jsonl, mt5/checkpoint/config.json, mt5/checkpoint/generation_config.json, mt5/checkpoint/model.safetensors, mt5/checkpoint/special_tokens_map.json, mt5/checkpoint/spiece.model, mt5/checkpoint/tokenizer.json, mt5/checkpoint/tokenizer_config.json, mt5/metrics.json, mt5/predictions.jsonl.


In [1]:
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","-q","datasets>=3.0","transformers>=4.55,<5","sentencepiece>=0.2","accelerate>=1.0"])
"""Deterministic English-to-French OPUS Books preprocessing."""
import argparse
import hashlib
import json
import random
import re
import unicodedata
from pathlib import Path

DATASET = "Helsinki-NLP/opus_books"
CONFIG = "en-fr"
REVISION = "main"


def clean(value):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", value or "")).strip()


def prepare(rows, train_size=4000, val_size=200, test_size=200, seed=4403):
    """Deduplicate on source, then shuffle; source strings cannot cross splits."""
    unique = {}
    rejected = {"empty_or_length": 0, "length_ratio": 0, "duplicate_source": 0}
    for row in rows:
        pair = row["translation"]
        en, fr = clean(pair["en"]), clean(pair["fr"])
        if not (10 <= len(en) <= 300 and 10 <= len(fr) <= 350):
            rejected["empty_or_length"] += 1
            continue
        ratio = len(fr) / len(en)
        if not 0.45 <= ratio <= 2.2:
            rejected["length_ratio"] += 1
            continue
        key = en.casefold()
        if key in unique:
            rejected["duplicate_source"] += 1
            continue
        uid = hashlib.sha256((en + "\0" + fr).encode()).hexdigest()[:16]
        unique[key] = {"id": uid, "en": en, "fr": fr}
    pairs = list(unique.values())
    random.Random(seed).shuffle(pairs)
    n = train_size + val_size + test_size
    if len(pairs) < n:
        raise ValueError(f"Need {n} clean unique pairs; found {len(pairs)}")
    picked = pairs[:n]
    splits = {
        "train": picked[:train_size],
        "validation": picked[train_size:train_size + val_size],
        "test": picked[train_size + val_size:],
    }
    return splits, {"raw_rows": len(rows), "clean_unique_rows": len(pairs),
                    "rejected": rejected, "seed": seed,
                    "sizes": {k: len(v) for k, v in splits.items()},
                    "source": f"https://huggingface.co/datasets/{DATASET}/tree/{REVISION}/{CONFIG}",
                    "language_pair": "en-fr", "normalization": "Unicode NFKC and whitespace collapse",
                    "filters": "English 10-300 chars, French 10-350 chars, length ratio 0.45-2.2",
                    "deduplication": "casefolded English source; first occurrence retained"}


def write_splits(splits, manifest, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    for name, items in splits.items():
        with (output / f"{name}.jsonl").open("w", encoding="utf-8") as f:
            for item in items:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
    (output / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--output", default="data/processed")
    p.add_argument("--train-size", type=int, default=4000)
    p.add_argument("--val-size", type=int, default=200)
    p.add_argument("--test-size", type=int, default=200)
    p.add_argument("--seed", type=int, default=4403)
    a = p.parse_args()
    from datasets import load_dataset
    rows = load_dataset(DATASET, CONFIG, split="train", revision=REVISION)
    splits, manifest = prepare(rows, a.train_size, a.val_size, a.test_size, a.seed)
    write_splits(splits, manifest, a.output)
    print(json.dumps(manifest, indent=2))



sys.argv=['prepare_data']
main()
"""Fine-tune mT5-small and translate the fixed English-French test split."""
import argparse
import json
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_ID = "google/mt5-small"
PROMPT = "translate English to French: "


def read(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()]


def run(data_dir, output, epochs=1, batch_size=8, seed=4403):
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required for this experiment")
    torch.manual_seed(seed)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    train, val, test = (read(Path(data_dir) / (name + ".jsonl"))
                        for name in ("train", "validation", "test"))
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).cuda()
    model.gradient_checkpointing_enable()
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

    def collate(rows):
        enc = tokenizer([PROMPT + r["en"] for r in rows], truncation=True,
                        max_length=128, padding=True, return_tensors="pt")
        labels = tokenizer(text_target=[r["fr"] for r in rows], truncation=True,
                           max_length=128, padding=True, return_tensors="pt")["input_ids"]
        labels[labels == tokenizer.pad_token_id] = -100
        return {**{k: v.cuda() for k, v in enc.items()}, "labels": labels.cuda()}

    history = []
    start = time.perf_counter()
    best = float("inf")
    for epoch in range(epochs):
        model.train()
        losses = []
        loader = DataLoader(train, batch_size=batch_size, shuffle=True, collate_fn=collate)
        for batch in loader:
            optimizer.zero_grad(set_to_none=True)
            # mT5 can overflow in fp16 on T4; float32 keeps the loss finite.
            loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite mT5 loss at epoch {epoch + 1}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())
        model.eval()
        vals = []
        with torch.inference_mode():
            for batch in DataLoader(val, batch_size=batch_size, collate_fn=collate):
                vals.append(model(**batch).loss.item())
        row = {"epoch": epoch + 1, "train_loss": sum(losses) / len(losses),
               "validation_loss": sum(vals) / len(vals)}
        print(row, flush=True)
        history.append(row)
        if row["validation_loss"] < best:
            best = row["validation_loss"]
            model.save_pretrained(output / "checkpoint", safe_serialization=True)
            tokenizer.save_pretrained(output / "checkpoint")
    training_seconds = time.perf_counter() - start
    model = AutoModelForSeq2SeqLM.from_pretrained(output / "checkpoint", torch_dtype=torch.float32).cuda().eval()
    torch.cuda.synchronize()
    start = time.perf_counter()
    predictions = []
    with torch.inference_mode():
        for i in range(0, len(test), batch_size):
            enc = tokenizer([PROMPT + r["en"] for r in test[i:i + batch_size]],
                            truncation=True, max_length=128, padding=True,
                            return_tensors="pt").to("cuda")
            ids = model.generate(**enc, max_new_tokens=96, do_sample=False,
                                 num_beams=1)
            predictions += tokenizer.batch_decode(ids, skip_special_tokens=True)
    torch.cuda.synchronize()
    inference_seconds = time.perf_counter() - start
    with (output / "predictions.jsonl").open("w", encoding="utf-8") as f:
        for row, prediction in zip(test, predictions):
            f.write(json.dumps({**row, "prediction": prediction}, ensure_ascii=False) + "\n")
    metrics = {"model": "mt5", "model_id": MODEL_ID, "architecture": "encoder-decoder",
               "parameters": sum(p.numel() for p in model.parameters()),
               "training_seconds": training_seconds, "inference_seconds": inference_seconds,
               "test_count": len(test), "history": history, "epochs": epochs,
               "decoding": "greedy, max 96 new tokens", "device": torch.cuda.get_device_name(0)}
    (output / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    return metrics



print(json.dumps(run('data/processed', '/kaggle/working/mt5'), indent=2), flush=True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.


gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


Generating train split: 100%|██████████| 127085/127085 [00:00<00:00, 526605.90 examples/s]


{


... (46 outputs left out)

/usr/local/lib/python3.13/dist-packages/nbconvert/filters/filter_links.py:36: SyntaxWarning: invalid escape sequence '\_'


  text = re.sub(r'_', '\_', text) # Escape underscores in display text


[NbConvertApp] Converting notebook __script__.ipynb to html


[NbConvertApp] Writing 316759 bytes to __results__.html
